# 📋 Notebook 6 — Reporte Final del Proyecto

**Objetivo:** Consolidar todos los resultados de los 4 pipelines en un resumen ejecutivo  

---
Este notebook carga todos los outputs generados por Kedro y los presenta
de forma integrada con gráficos y conclusiones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Librerías cargadas')

## 1. Cargar todos los outputs de Kedro

In [ ]:
# Resultados de clasificación (Pipeline 3)
metricas = pd.read_csv('../data/07_model_output/classification_metrics.csv')

# Resultados de clustering (Pipeline 4)
clusters = pd.read_csv('../data/07_model_output/clustering_results.csv')

# Reporte final (Pipeline 5)
reporte = pd.read_csv('../data/08_reporting/final_report.csv')

# Dataset limpio
df_clean = pd.read_csv('../data/02_intermediate/clean_attributes.csv')

print('Archivos cargados:')
print(f'  ✅ classification_metrics: {metricas.shape}')
print(f'  ✅ clustering_results:     {clusters.shape}')
print(f'  ✅ final_report:           {reporte.shape}')
print(f'  ✅ clean_attributes:       {df_clean.shape}')

## 2. Resumen Ejecutivo

In [ ]:
print('=' * 55)
print('     REPORTE EJECUTIVO — FACE ATTRIBUTES PROJECT')
print('=' * 55)

for _, row in reporte.iterrows():
    print(f"  [{row['seccion']:15s}] {row['metrica']:25s}: {row['valor']}")

print('=' * 55)
print(f'Dataset original:  30.141 imágenes de caras')
print(f'Dataset limpio:    {len(df_clean):,} imágenes (sin duplicados)')
print(f'Atributos totales: 33 (binarios: 0 o 1)')
print(f'Pipelines Kedro:   4 pipelines + 1 reporting')
print('=' * 55)

## 3. Dashboard Final — Todos los resultados en un gráfico

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── Panel 1: Comparación de modelos ──
ax1 = fig.add_subplot(gs[0, 0:2])
x = np.arange(len(metricas))
width = 0.2
metricas_cols = ['accuracy','precision','recall','f1_score']
colores = ['#3498DB','#E74C3C','#2ECC71','#F39C12']
for i, (col, color) in enumerate(zip(metricas_cols, colores)):
    if col in metricas.columns:
        ax1.bar(x + i*width, metricas[col], width, label=col.replace('_',' ').title(), color=color, alpha=0.85)
ax1.set_xticks(x + width*1.5)
ax1.set_xticklabels(metricas['modelo'] if 'modelo' in metricas.columns else metricas.index, fontsize=9)
ax1.set_ylim(0, 1.05)
ax1.set_title('Pipeline 3 — Comparación de Modelos', fontweight='bold')
ax1.legend(fontsize=8)

# ── Panel 2: Distribución de clusters ──
ax2 = fig.add_subplot(gs[0, 2])
if 'cluster_size' in clusters.columns:
    sizes = clusters['cluster_size'].values
    labels_pie = [f'Cluster {i}\n({s:,})' for i, s in enumerate(sizes)]
    ax2.pie(sizes, labels=labels_pie, autopct='%1.1f%%',
            colors=['#E74C3C','#3498DB','#2ECC71'], startangle=90)
ax2.set_title('Pipeline 4 — Distribución\nde Clusters', fontweight='bold')

# ── Panel 3: Dataset antes vs después de limpieza ──
ax3 = fig.add_subplot(gs[1, 0])
ax3.bar(['Crudo', 'Limpio'], [30141, len(df_clean)],
        color=['#E74C3C','#2ECC71'], edgecolor='black')
ax3.set_title('Pipeline 1 — Limpieza\nde Datos', fontweight='bold')
ax3.set_ylabel('Filas')
for i, v in enumerate([30141, len(df_clean)]):
    ax3.text(i, v+200, f'{v:,}', ha='center', fontweight='bold')

# ── Panel 4: Balance del target ──
ax4 = fig.add_subplot(gs[1, 1])
conteo = df_clean['attractive'].value_counts()
ax4.bar(['No Atractivo (0)', 'Atractivo (1)'], conteo.values,
        color=['#FF6B6B','#4ECDC4'], edgecolor='black')
ax4.set_title('Pipeline 2 — Balance\ndel Target', fontweight='bold')
ax4.set_ylabel('Cantidad')
for i, v in enumerate(conteo.values):
    ax4.text(i, v+50, f'{v:,}', ha='center', fontweight='bold')

# ── Panel 5: Features seleccionadas ──
ax5 = fig.add_subplot(gs[1, 2])
ax5.pie([26, 11], labels=['Seleccionadas\n(26)', 'Descartadas\n(11)'],
        colors=['#2ECC71','#E74C3C'], autopct='%1.0f%%', startangle=90)
ax5.set_title('Pipeline 2 — Selección\nde Features', fontweight='bold')

fig.suptitle('🎯 Dashboard Final — Face Attributes Kedro Project',
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig('../data/08_reporting/dashboard_final.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Dashboard guardado en data/08_reporting/dashboard_final.png')

## 4. Tabla Final de Métricas por Pipeline

In [ ]:
resumen = pd.DataFrame([
    {'Pipeline': '1 - Data Processing',     'Tarea': 'Limpieza',        'Resultado': '30.141 → 13.141 filas (eliminó 17.000 duplicados)'},
    {'Pipeline': '2 - Feature Engineering', 'Tarea': 'Nuevas features', 'Resultado': '33 → 38 columnas (+5 scores compuestos)'},
    {'Pipeline': '2 - Feature Engineering', 'Tarea': 'Selección',       'Resultado': '37 → 26 features (corr ≥ 0.05)'},
    {'Pipeline': '2 - Feature Engineering', 'Tarea': 'Split',           'Resultado': '10.512 train / 2.629 test (80/20)'},
    {'Pipeline': '3 - Classification',      'Tarea': 'Mejor modelo',    'Resultado': 'Logistic Regression — Accuracy: 78.3%, F1: 0.618'},
    {'Pipeline': '4 - Clustering',          'Tarea': 'K-Means k=3',     'Resultado': '3 clusters — Silhouette: 0.131'},
    {'Pipeline': '4 - Clustering',          'Tarea': 'Perfil Cluster 0','Resultado': '23% del dataset — Cara femenina (cabello, maquillaje)'},
    {'Pipeline': '4 - Clustering',          'Tarea': 'Perfil Cluster 1','Resultado': '15% del dataset — Cara envejecida (arrugas, doble mentón)'},
    {'Pipeline': '4 - Clustering',          'Tarea': 'Perfil Cluster 2','Resultado': '61% del dataset — Cara masculina adulta (pelo negro, barba)'},
])

print('=== RESUMEN COMPLETO DEL PROYECTO ===')
resumen

## ✅ Conclusiones Generales

1. **El dataset tiene 17.000 duplicados** (56% de las filas), lo cual se limpió en el Pipeline 1.

2. **Predecir attractiveness es posible** con ~78% de accuracy usando solo atributos binarios, aunque el F1 (~0.62) indica que el modelo tiene margen de mejora con más datos o features.

3. **Existen 3 perfiles naturales** de caras en el dataset: cara femenina joven, cara envejecida, y cara masculina adulta (que representa el 61% del dataset).

4. **Kedro organizó el proyecto** en pipelines independientes y reproducibles, con todos los parámetros centralizados en `parameters.yml` y los datasets en `catalog.yml`.

---
*Proyecto desarrollado con Kedro 1.4.0 | Dataset: Attributes.csv (Face Attributes)*